# JAX Vision-Range Curriculum

Train a fixed 50x50 AntByte task while shrinking the actor vision window from 51x51 to 3x3.

In [11]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ant_byte_env import notebook_workflows as workflows

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status

Restart the kernel before rerunning training; JAX was already imported.


{'jax_already_imported': True,
 'jax_preallocate': 'false',
 'jax_memory_fraction': '0.35',
 'jax_allocator': 'platform',
 'memory_trimmed': True,
 'disk_free_gb': 26.42,
 'disk_used_percent': 76.8,
 'current_pid': 13267,
 'safe_cleanup_candidate_count': 0,
 'safe_cleanup_candidate_gb': 0.0,
 'top_memory_processes': [{'pid': 4481,
   'ppid': 2766,
   'rss_mb': 1577.1,
   'command': '/snap/firefox/7901/usr/lib/firefox/firefox',
   'connection_file': None,
   'is_current_process': False,
   'is_notebook_kernel': False},
  {'pid': 13267,
   'ppid': 12088,
   'rss_mb': 1229.5,
   'command': '/home/narf/miniconda3/envs/cool-antz/bin/python -m ipykernel_launcher --f=/run/user/1000/jupyter/runtime/kernel-v3bca49abf50072073278c980fba9eb3bc88946b5f.json',
   'connection_file': '/run/user/1000/jupyter/runtime/kernel-v3bca49abf50072073278c980fba9eb3bc88946b5f.json',
   'is_current_process': True,
   'is_notebook_kernel': True},
  {'pid': 13807,
   'ppid': 4647,
   'rss_mb': 780.9,
   'command': '

In [12]:
import jax

from ant_byte_env.experiments import load_experiment_config
from ant_byte_env.training.jax_mappo import runner as jax_runner

EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "vision_range_curriculum.json"
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "vision_range_curriculum"
experiment = load_experiment_config(EXPERIMENT_CONFIG)
TRAINING_ARGS = dict(experiment.args)
VISION_RADII = tuple(experiment.metadata["vision_radii"])
GLOBAL_UPDATE_CAP = int(experiment.metadata["global_update_cap"])
RENDER_MAX_FRAMES = int(experiment.metadata["render_max_frames"])
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
UPDATE_TIMESTEPS_PER_STAGE = int(TRAINING_ARGS["num_envs"]) * int(TRAINING_ARGS["num_steps"])
COMMON_ARGS = workflows.config_common_args(
    TRAINING_ARGS,
    exclude=workflows.VISION_RANGE_ARG_EXCLUDES,
)

{"backend": jax.default_backend(), "vision_radii": VISION_RADII}

{'backend': 'gpu', 'vision_radii': (10,)}

In [13]:
vision_result = workflows.run_vision_range_curriculum(
    vision_radii=VISION_RADII,
    run_dir=RUN_DIR,
    common_args=COMMON_ARGS,
    experiment_name=experiment.name,
    update_timesteps_per_stage=UPDATE_TIMESTEPS_PER_STAGE,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    max_render_frames=RENDER_MAX_FRAMES,
    tile_size=ROLLOUT_TILE_SIZE,
)
vision_result["final_checkpoint"]

Training vision stage 1/1: 21x21


21x21: 5/5 updates |██████████| 01:52<00:00 , loss=-0.002, ret=-289.670, ret_avg=-78.295


PosixPath('/home/narf/Desktop/Facultad/RL/cool-antz-nuevo/cool-antz/runs/notebooks/vision_range_curriculum/21x21/checkpoints/model.pkl')

In [14]:
from ant_byte_env.rendering import render_checkpoint

LONG_RENDER_MAX_FRAMES = 2_000
LONG_RENDER_STAGE_RADIUS = int(VISION_RADII[-1])
LONG_RENDER_STAGE_NAME = f"{workflows.vision_side(LONG_RENDER_STAGE_RADIUS)}x{workflows.vision_side(LONG_RENDER_STAGE_RADIUS)}"
LONG_RENDER_OUTPUT = RUN_DIR / "media" / f"vision_{LONG_RENDER_STAGE_NAME}_long_{LONG_RENDER_MAX_FRAMES}frames.gif"

LONG_RENDER_CHECKPOINT = None
if "vision_result" in globals():
    LONG_RENDER_CHECKPOINT = vision_result.get("final_checkpoint")
if LONG_RENDER_CHECKPOINT is None:
    LONG_RENDER_CHECKPOINT = RUN_DIR / LONG_RENDER_STAGE_NAME / "checkpoints" / "model.pkl"
LONG_RENDER_CHECKPOINT = Path(LONG_RENDER_CHECKPOINT)

render_checkpoint(
    LONG_RENDER_CHECKPOINT,
    LONG_RENDER_OUTPUT,
    backend="jax",
    seed_offset=workflows.NOTEBOOK_ROLLOUT_SEED_OFFSET + len(VISION_RADII) - 1,
    reuse_existing=False,
    max_frames=LONG_RENDER_MAX_FRAMES,
    tile_size=ROLLOUT_TILE_SIZE,
    policy_temperature=workflows.NOTEBOOK_ROLLOUT_POLICY_TEMPERATURE,
)

PosixPath('/home/narf/Desktop/Facultad/RL/cool-antz-nuevo/cool-antz/runs/notebooks/vision_range_curriculum/media/vision_21x21_long_2000frames.gif')

In [15]:
# import matplotlib.pyplot as plt

# metrics = vision_result["stage_metrics"]
# MEDIA_DIR = RUN_DIR / "media"
# MEDIA_DIR.mkdir(parents=True, exist_ok=True)
# updates = list(range(1, len(metrics) + 1))

# def metric_values(name):
#     return [float(row[name]) for row in metrics if name in row]

# fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
# for name in ("episode_return", "env_return"):
#     series = metric_values(name)
#     if series:
#         axes[0].plot(updates[: len(series)], series, label=name)
# for name in ("loss", "policy_loss", "value_loss"):
#     series = metric_values(name)
#     if series:
#         axes[1].plot(updates[: len(series)], series, label=name)
# for axis in axes:
#     axis.grid(True, alpha=0.3)
#     axis.legend()
# axes[0].set_ylabel("return")
# axes[1].set_ylabel("loss")
# axes[1].set_xlabel("curriculum update")
# fig.tight_layout()
# plot_path = MEDIA_DIR / "reward_loss.png"
# fig.savefig(plot_path, dpi=160)
# plot_path